In [2]:
import sqlite3
import pandas as pd
from pathlib import Path

db_path = r"C:\Users\akmaluddin.hasni\AppData\Roaming\DBeaverData\workspace6\.metadata\sample-database-sqlite-1\Chinook.db"
conn = sqlite3.connect(db_path)


In [3]:
import sqlite3
import pandas as pd

conn = sqlite3.connect(
    r"C:\Users\akmaluddin.hasni\AppData\Roaming\DBeaverData\workspace6\.metadata\sample-database-sqlite-1\Chinook.db"
)

tables = pd.read_sql("""
    SELECT name
    FROM sqlite_master
    WHERE type = 'table'
      AND name NOT LIKE 'sqlite_%'
    ORDER BY name
""", conn)

dataframes_SQLite = {}

for table_name in tables["name"]:
    df = pd.read_sql(f'SELECT * FROM "{table_name}"', conn)
    dataframes_SQLite[table_name] = df



In [ ]:
sql_path = Path("query1.sql")
query = sql_path.read_text(encoding="utf-8")
df = pd.read_sql_query(query, conn)
print(df)

In [ ]:
base_path = Path(".")

# Load and execute SQL files, then read results into DataFrames

# 1. Single-SELECT files can be loaded directly with read_sql_query
query_01 = (base_path / "01_data_cleaning.sql").read_text(encoding="utf-8")
df_01_data_cleaning = pd.read_sql_query(query_01, conn)

query_02 = (base_path / "02_data_transformation.sql").read_text(encoding="utf-8")
df_02_data_transformation = pd.read_sql_query(query_02, conn)

query_05 = (base_path / "05_sales_report.sql").read_text(encoding="utf-8")
df_05_sales_report = pd.read_sql_query(query_05, conn)

# 2. Multi-statement files need executescript first, then a separate SELECT

script_03 = (base_path / "03_create_summary_table.sql").read_text(encoding="utf-8")
conn.executescript(script_03)
df_03_create_summary_table = pd.read_sql_query(
    "SELECT * FROM CustomerSpendingSummary ORDER BY TotalSpent DESC",
    conn
)

script_04 = (base_path / "04_update_example.sql").read_text(encoding="utf-8")
conn.executescript(script_04)
df_04_update_example = pd.read_sql_query(
    """
    SELECT CustomerId, FirstName, LastName, Company
    FROM Customer_Copy
    WHERE Company = 'Individual Customer'
    """,
    conn
)

script_06 = (base_path / "06_create_view.sql").read_text(encoding="utf-8")
conn.executescript(script_06)
df_06_create_view = pd.read_sql_query(
    "SELECT * FROM vw_customer_invoice_summary ORDER BY Total DESC",
    conn
)

In [ ]:
conn.close()